## Dataset downlaod to train 1) MLP, and 2) LSTM with MLP already trained

### Imports

In [ ]:
import os
import shutil
import numpy as np
# import matplotlib.pyplot as plt
import pandas as pd
# from sklearn.preprocessing import LabelEncoder
# from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, recall_score, classification_report, precision_score
import torch
# import torch.nn as nn
# from torch.utils.data import TensorDataset  # for saving after removing features
import kagglehub

from torch.utils.data import TensorDataset
from sklearn.preprocessing import LabelEncoder

### Background - already existing functions

In [ ]:
# check if GPU available
gpu_av=torch.cuda.is_available()

# for reproducibility
SEED = 42
torch.manual_seed(SEED)
print("GPU available:", gpu_av)

if gpu_av:
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

print(f"CPU cores: {os.cpu_count()}")


GPU available: False
CPU cores: 2


In [2]:
# SAME functions as Project.ipynb / data_loader.py latest versions (WORKING HERE FIRST)

def download_dataset(
    year_start,
    year_end_exd,  # end year (excluded)
    origin_path="flnny123/mfddmulti-modal-flight-delay-dataset/versions/4",
    mode="tabular",  # or "sequential" for pre-made chains
    output_dir_seq="data/chain/",
):

    dest_paths = []
    for year in range(year_start, year_end_exd):
        print(f"Downloading year {year} data...")
        if mode == "tabular":  # tabular dataset download
            origin_path_year = (
                "Aeolus/Flight_Tab/flight_with_weather_" + str(year) + ".csv"
            )
            dest_path_year = kagglehub.dataset_download(
                origin_path, path=origin_path_year
            )
            dest_paths.append(dest_path_year)

        # dest_path_year = dest_path+'flight_with_weather_'+str(year)+'.csv' # destination path
        elif mode == "sequential":  # sequential (chains) dataset download
            for split in ["train", "val", "test"]:
                if year == 2024:
                    origin_path_year_split = f"Aeolus/Flight_chain/chain_data_{year}/flight_chain_{split}_{year}.pt"
                # different naming convention in original dataset
                else:
                    origin_path_year_split = f"Aeolus/Flight_chain/chain_data_{year}/{split}_flight_chain_{year}.pt"

                final_path = os.path.join(
                    output_dir_seq + str(year), f"{split}_flight_chain_{year}.pt"
                )
                if os.path.exists(final_path):
                    print(f"Path {final_path} already exists! Skipping it")
                    continue

                dest_path_year = kagglehub.dataset_download(
                    origin_path, path=origin_path_year_split
                )
                os.makedirs(output_dir_seq + str(year), exist_ok=True)
                shutil.move(dest_path_year, final_path)
                dest_path_year = final_path
                dest_paths.append(final_path)

                print(f"(File(s) available at {dest_path_year}).")

            

    return dest_paths


def load_dataset_pytorch(year_start, year_end, file_path="data/chain/"):
    """
    Load sequential chain datasets for years in [year_start, year_end) and merge
    all years into a single dataset per split (train, val, test).
    """
    split_types = ["train", "val", "test"]
    loaded_data = {split: [] for split in split_types}  # store lists of tensors

    for year in range(year_start, year_end):
        for split in split_types:
            full_file_path = file_path + f"{year}/{split}_flight_chain_{year}.pt"
            dataset = torch.load(full_file_path, weights_only=False)
            # n_samples_to_inspect = 3

            # for i in range(n_samples_to_inspect):
            #     sample = dataset[i]
            #     print(f"--- Sample {i} ---")
            #     for j, tensor in enumerate(sample):
            #         print(f"  tensors[{j}] shape: {tensor.shape}, dtype: {tensor.dtype}")
            #         print(f"  tensors[{j}] values:\n{tensor}\n")
            #     print("=" * 60)

            # Slice dense tensor to remove FLIGHTS (last column)
            dense = dataset.tensors[0]  # [N, seq_len, 7]
            dense = dense[:, :, :-1].clone()  # [N, seq_len, 6]

            # Rebuild dataset (other tensors unchanged)
            processed = TensorDataset(dense, *dataset.tensors[1:])
            loaded_data[split].append(processed)
            print(f"--- Read file: (split: {split}, year: {year}) ---")

    # Concatenate all years for each split
    merged = {}
    for split in split_types:
        # Gather all tensors from each dataset in the list
        all_dense = torch.cat([ds.tensors[0] for ds in loaded_data[split]], dim=0)
        all_sparse = torch.cat([ds.tensors[1] for ds in loaded_data[split]], dim=0)
        all_labels = torch.cat([ds.tensors[2] for ds in loaded_data[split]], dim=0)
        all_lens = torch.cat([ds.tensors[3] for ds in loaded_data[split]], dim=0)
        all_delays = torch.cat([ds.tensors[4] for ds in loaded_data[split]], dim=0)

        merged[split] = TensorDataset(
            all_dense, all_sparse, all_labels, all_lens, all_delays
        )
        print(f"--- Merged {split}: {all_dense.shape[0]} samples across years ---")

    return merged



In [3]:

def clean_dataframe(
    df,
    int_type="int32",
    float_type="float32",
    cache_bool=False,
):  # cache=False to save memory at the cost of speed
    # type conversions
    for col in DATE_COLS + DATETIME_COLS:
        df[col] = pd.to_datetime(df[col], format="%Y-%m-%d %H:%M:%S", cache=cache_bool)
    for col in DATE_COLS:
        df[col] = df[col].dt.normalize()  # keep date only, no time (cleaner)
    for col in TIMEDELTA_MINS_COLS:
        df[col] = pd.to_timedelta(df[col], unit="m")
    df[INT_COLS] = df[INT_COLS].astype(int_type)
    df[STR_COLS] = df[STR_COLS].astype("str")
    df[FLOAT_COLS] = df[FLOAT_COLS].astype(float_type)  # limit precision??

    # drop all rows containing NaNs and keep track of them
    n_rows_before = len(df)
    df.dropna(subset=["D_TEMP", "D_PRCP", "D_WSPD"], inplace=True)
    n_rows_after = len(df)
    n_rows_dropped = n_rows_before - n_rows_after
    print(f"Dropped {n_rows_dropped} rows because of NaNs.")
    nan_bool = bool(np.any(df.isna().sum() > 0))
    print(f"Any NaNs remaining in numerical data: {nan_bool}.")

    return df

In [ ]:
def create_flight_chains(df):
    df_sorted = df.sort_values(
        by=["OP_CARRIER", "OP_CARRIER_FL_NUM", "FL_DATE", "CRS_DEP_TIME"]
    )
    grouped = df_sorted.groupby(
        ["OP_CARRIER", "OP_CARRIER_FL_NUM", "FL_DATE"]
    )  # unique aircraft identifiers
    res = {name: group for name, group in grouped}  # dict
    return res


# Truncation/padding for single chain
def adjust_sequence(data, max_len):
    if len(data) < max_len:
        pad_shape = (max_len - len(data), data.shape[1])
        return torch.cat([data, torch.zeros(pad_shape, dtype=data.dtype)], dim=0)
    return data[:max_len]


def process_all_chains(
    flight_chains,
    target_columns,
    dense_feat_cols,
    sparse_feat_cols,
    max_sequence_length,
    year,  # for saving to torch dataset with proper name
):
    processed = []
    for name, chain in flight_chains.items():
        dense_tensors = [
            torch.tensor(
                chain[name].fillna(0).values, dtype=torch.float32
            )  # handle NaNs; NNs expect float
            for name in dense_feat_cols
        ]
        dense_feat = torch.stack(
            dense_tensors, dim=1
        )  # stacks columns horizontally, resulting shape (seq_len, num_dense_features)

        # feature engineering
        chain["MONTH"] = (chain["MONTH"] - 1).astype(
            np.int16
        )  # standard 0-based embedding + store as int16 to save memory
        chain["DAY_OF_WEEK"] = (chain["DAY_OF_WEEK"] - 1).astype(np.int16)

        sparse_tensors = [
            torch.tensor(
                chain[name].values.astype(np.int16), dtype=torch.int16
            )  # categorical columns should not have NAs
            for name in sparse_feat_cols
        ]
        sparse_feat = torch.stack(sparse_tensors, dim=1)

        # labels for binary classification + full delays
        # have to convert to minutes since it's a timedelta object
        delays = torch.tensor(
            (
                chain[target_columns].apply(lambda s: s.dt.total_seconds() / 60)
            ).values.astype(np.int16),
            dtype=torch.int16,
        )

        labels = torch.tensor(
            np.column_stack(
                (
                    (chain["ARR_DELAY"].dt.total_seconds() / 60 > 15).astype(
                        np.int8
                    ),  # definition of delay if more than 15 min (for binary classification)
                    (chain["ARR_DELAY"].dt.total_seconds() / 60 > 15).astype(np.int8),
                )
            ),
            dtype=torch.int8,
        )

        # Sequence length processing
        valid_len = min(
            len(dense_feat), max_sequence_length
        )  # stores the actual length before padding (but after truncation). Useful later eg in loss calculation
        dense_feat = adjust_sequence(dense_feat, max_sequence_length)
        sparse_feat = adjust_sequence(sparse_feat, max_sequence_length)
        labels = adjust_sequence(labels, max_sequence_length)
        delays = adjust_sequence(delays, max_sequence_length)

        processed.append((dense_feat, sparse_feat, labels, valid_len, delays))
    return processed


def create_dataset(processed_data):
    dense = torch.stack([item[0] for item in processed_data])
    sparse = torch.stack([item[1] for item in processed_data])
    labels = torch.stack([item[2] for item in processed_data])
    valid_lens = torch.tensor([item[3] for item in processed_data], dtype=torch.long)
    delays = torch.stack([item[4] for item in processed_data])
    return TensorDataset(dense, sparse, labels, valid_lens, delays)


# IMPORTANT NOTE: this modifies dataframe in-place
def prepare_data(
    df,
    year,
    mode="tabular",  # tabular or sequential
    target_columns=["DEP_DELAY", "ARR_DELAY"],  # in minutes
    time_of_prediction="departure",
    seed=42,
    max_sequence_length=6,
    train_frac=0.6,
    valid_frac=0.2,
):  # departure or arrival (for tabular mode)

    print(f"Start processing data for {year}...")
    start_time = time.time()
    if mode == "tabular":
        df = remove_outliers_percentile(df, target_columns)
        df = df.dropna(subset=["FL_DATE"])
        df["FL_YEAR"] = df["FL_DATE"].dt.year  # flight year...
        df.drop(columns=["FL_DATE"], inplace=True)  # ... instead of full date

        time_columns = [
            "CRS_DEP_TIME",
            "DEP_TIME",
            "WHEELS_OFF",
            "WHEELS_ON",
            "CRS_ARR_TIME",
            "ARR_TIME",
        ]
        for col in time_columns:
            df[col + "_MIN"] = df[col].dt.hour * 60 + df[col].dt.minute
            # convert time columns in minutes since midnight...
        df.drop(columns=time_columns, inplace=True)  # ... and drop the originals

        categorical_columns = [
            "OP_CARRIER",
            "OP_CARRIER_FL_NUM",
            "FL_YEAR",
            "MONTH",
            "DAY_OF_MONTH",  # unique identifier for any day;
            # scheduled?? if not, minor leak
            "ORIGIN",
            "DEST",
        ]

        # info available before actual departure time
        continuous_columns = [
            "CRS_DEP_TIME_MIN",
            "CRS_ARR_TIME_MIN",
            "FLIGHTS",
            "O_TEMP",
            "O_PRCP",
            "O_WSPD",
            "D_TEMP",
            "D_PRCP",
            "D_WSPD",
            "O_LATITUDE",
            "O_LONGITUDE",
            "D_LATITUDE",
            "D_LONGITUDE",
        ]
        if time_of_prediction == "arrival":
            # additional info available before actual arrival time (but after actual departure)
            continuous_columns = continuous_columns + [
                "DEP_TIME_MIN",
                "WHEELS_OFF_MIN",  # could also try adding "TAXI_IN"?
            ]
            target_cols = ["ARR_DELAY"]
        else:
            target_cols = target_columns
        df = df[target_cols + categorical_columns + continuous_columns]
        return df

    elif mode == "sequential":
        # feature engineering
        df["CRS_DEP_TIME_HOUR"] = df["CRS_DEP_TIME"].dt.hour.astype("int8")
        df["CRS_ARR_TIME_HOUR"] = df["CRS_ARR_TIME"].dt.hour.astype("int8")

        # encode categorical features
        encoder = LabelEncoder()
        df["OP_CARRIER"] = encoder.fit_transform(df["OP_CARRIER"])
        df["OP_CARRIER_FL_NUM"] = encoder.fit_transform(df["OP_CARRIER_FL_NUM"])

        # more feature engineering
        df["MONTH"] = df["FL_DATE"].dt.month
        df["DAY_OF_YEAR"] = df["FL_DATE"].dt.dayofyear

        dense_feat_cols = [  # numeric, all stored as floats;
            # relationship between them is quantitative (eg it makes sense to think that twice the flights will somehow have twice the impact on delay)
            "O_TEMP",
            "D_TEMP",  # assuming measured at departure??
            "O_PRCP",
            "D_PRCP",
            "O_WSPD",
            "D_WSPD",
            "FLIGHTS",
        ]

        sparse_feat_cols = [  # categorical, stored as ints
            "MONTH",
            "DAY_OF_WEEK",
            "CRS_ARR_TIME_HOUR",
            "CRS_DEP_TIME_HOUR",
            "ORIGIN_INDEX",
            "DEST_INDEX",
            "OP_CARRIER",
            "OP_CARRIER_FL_NUM",
        ]

        target_cols = target_columns

        # df = df[target_cols + ["DAY_OF_YEAR"] + ["FL_DATE"] + dense_feat_cols + sparse_feat_cols]
        # from here on we include train/test split (without leaking), torch implementation, etc.
        # (group by to create multiple chains each identified by starting day of year, etc)
        # and also time of prediction

        df = df.sort_values(by="FL_DATE").reset_index(drop=True)
        date_dim = (
            df[["FL_DATE"]].drop_duplicates()
        )  # selects FL_DATE column as a dataframe to get only unique dates
        # new columsna for unique identifier
        date_dim["DAY_OF_YEAR"] = date_dim["FL_DATE"].dt.dayofyear
        date_dim["MONTH"] = date_dim["FL_DATE"].dt.month

        rng = np.random.RandomState(seed=seed)

        train_days, valid_days, test_days = [], [], []
        for month in sorted(date_dim["MONTH"].unique()):  # 1, 2, 3, ...
            month_dates = date_dim[date_dim["MONTH"] == month]["DAY_OF_YEAR"]
            n_days = len(month_dates)
            if n_days < 3:
                raise Exception(
                    f"Allocated days less than three for {month} with this seed."
                )
            #    alloc_days = list(month_dates) * 3
            #    mandatory_days = alloc_days[:3]
            # else:
            #
            mandatory_days = rng.choice(month_dates, 3, replace=False)

            train_days.append(mandatory_days[0])
            valid_days.append(mandatory_days[1])
            test_days.append(mandatory_days[2])

            remaining_days = [d for d in month_dates if d not in mandatory_days]
            n_remaining = len(remaining_days)

            if n_remaining > 0:
                permuted = rng.permutation(remaining_days)
                split1 = int(round(n_remaining * train_frac))
                split2 = split1 + int(round(n_remaining * valid_frac))

                train_days.extend(permuted[:split1])
                valid_days.extend(permuted[split1:split2])
                test_days.extend(permuted[split2:])

        date_dim["SPLIT_TYPE"] = np.select(
            [
                date_dim["DAY_OF_YEAR"].isin(train_days),
                date_dim["DAY_OF_YEAR"].isin(valid_days),
                date_dim["DAY_OF_YEAR"].isin(test_days),
            ],
            ["train", "valid", "test"],
            default="undefined",  # in case something goes wrong, split type will not be automatically deduced
        )

        df = pd.merge(df, date_dim[["FL_DATE", "SPLIT_TYPE"]], on="FL_DATE", how="left")

        train_chains = create_flight_chains(df[df["SPLIT_TYPE"] == "train"])
        valid_chains = create_flight_chains(df[df["SPLIT_TYPE"] == "valid"])
        test_chains = create_flight_chains(df[df["SPLIT_TYPE"] == "test"])

        train_processed = process_all_chains(
            train_chains,
            target_columns=target_columns,
            dense_feat_cols=dense_feat_cols,
            sparse_feat_cols=sparse_feat_cols,
            max_sequence_length=max_sequence_length,
            year=year,
        )
        valid_processed = process_all_chains(
            valid_chains,
            target_columns=target_columns,
            dense_feat_cols=dense_feat_cols,
            sparse_feat_cols=sparse_feat_cols,
            max_sequence_length=max_sequence_length,
            year=year,
        )
        test_processed = process_all_chains(
            test_chains,
            target_columns=target_columns,
            dense_feat_cols=dense_feat_cols,
            sparse_feat_cols=sparse_feat_cols,
            max_sequence_length=max_sequence_length,
            year=year,
        )

        train_dataset = create_dataset(train_processed)
        valid_dataset = create_dataset(valid_processed)
        test_dataset = create_dataset(test_processed)

        # 5. Save results
        output_dir = f"processed_data_{year}"
        os.makedirs(output_dir, exist_ok=True)

        torch.save(
            train_dataset, os.path.join(output_dir, f"train_flight_chain_{year}.pt")
        )
        torch.save(
            valid_dataset, os.path.join(output_dir, f"val_flight_chain_{year}.pt")
        )
        torch.save(
            test_dataset, os.path.join(output_dir, f"test_flight_chain_{year}.pt")
        )

        elapsed = time.time() - start_time
        print(
            f"Finished processing data for {year}, time elapsed: {elapsed:.2f} seconds"
        )
        return f"Torch dataset available at relative path: {output_dir}"

In [5]:
def add_cyclic_time_features(df: pd.DataFrame) -> pd.DataFrame:
    """Hour of day and day of week as sine/cosine instead of raw integers:
    preserves continuity (23:59 close to 00:00)."""
    df = df.copy()
    minutes_in_day = 24 * 60

    df["dep_hour_sin"] = np.sin(2 * np.pi * df["CRS_DEP_TIME_MIN"] / minutes_in_day)
    df["dep_hour_cos"] = np.cos(2 * np.pi * df["CRS_DEP_TIME_MIN"] / minutes_in_day)
    df["arr_hour_sin"] = np.sin(2 * np.pi * df["CRS_ARR_TIME_MIN"] / minutes_in_day)
    df["arr_hour_cos"] = np.cos(2 * np.pi * df["CRS_ARR_TIME_MIN"] / minutes_in_day)

    date = pd.to_datetime(dict(year=df["FL_YEAR"], month=df["FL_MONTH"], day=df["FL_DAY"]))
    dow = date.dt.dayofweek
    df["dow_sin"] = np.sin(2 * np.pi * dow / 7)
    df["dow_cos"] = np.cos(2 * np.pi * dow / 7)
    df["month_sin"] = np.sin(2 * np.pi * df["FL_MONTH"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["FL_MONTH"] / 12)
    return df


def add_great_circle_distance(df: pd.DataFrame) -> pd.DataFrame:
    """Great-circle distance (Haversine, km) between origin and destination."""
    df = df.copy()
    R = 6371.0
    lat1, lon1 = np.radians(df["O_LATITUDE"]), np.radians(df["O_LONGITUDE"])
    lat2, lon2 = np.radians(df["D_LATITUDE"]), np.radians(df["D_LONGITUDE"])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    df["great_circle_km"] = R * 2 * np.arcsin(np.sqrt(a))
    return df


def add_airport_congestion(df: pd.DataFrame) -> pd.DataFrame:
    """Congestion proxy: number of flights scheduled from the same origin
    airport, on the same day, within the same 2-hour window. Based only on
    SCHEDULED times -> no leakage, safe to compute even before the temporal
    split."""   
    df = df.copy()
    df["dep_hour_bucket"] = (df["CRS_DEP_TIME_MIN"] // 120).astype(int)
    congestion = (
        df.groupby(["ORIGIN_INDEX", "FL_YEAR", "FL_MONTH", "FL_DAY", "dep_hour_bucket"])
        .size().rename("origin_congestion_2h").reset_index()
    )
    df = df.merge(congestion, on=["ORIGIN_INDEX", "FL_YEAR", "FL_MONTH", "FL_DAY", "dep_hour_bucket"], how="left")
    return df


def add_binary_target(df: pd.DataFrame, threshold: int = 15) -> pd.DataFrame:
    """Binary target aligned with the official Aeolus benchmark (15-min threshold)."""
    df = df.copy()
    df["ARR_DELAY_BIN"] = (df["ARR_DELAY"] > threshold).astype(int)
    df["DEP_DELAY_BIN"] = (df["DEP_DELAY"] > threshold).astype(int)
    return df


def add_all_features(df: pd.DataFrame, delay_threshold: int = 15) -> pd.DataFrame:
    """Versione memory-safe: una sola copia iniziale, poi tutte le feature
    vengono aggiunte in-place sullo stesso oggetto, invece di incatenare
    funzioni che fanno ciascuna un .copy() completo del dataframe."""
    df = df.copy()  
    
    minutes_in_day = 24 * 60
    df["dep_hour_sin"] = np.sin(2 * np.pi * df["CRS_DEP_TIME_MIN"] / minutes_in_day)
    df["dep_hour_cos"] = np.cos(2 * np.pi * df["CRS_DEP_TIME_MIN"] / minutes_in_day)
    df["arr_hour_sin"] = np.sin(2 * np.pi * df["CRS_ARR_TIME_MIN"] / minutes_in_day)
    df["arr_hour_cos"] = np.cos(2 * np.pi * df["CRS_ARR_TIME_MIN"] / minutes_in_day)

    date = pd.to_datetime(dict(year=df["FL_YEAR"], month=df["FL_MONTH"], day=df["FL_DAY"]))
    dow = date.dt.dayofweek
    df["dow_sin"] = np.sin(2 * np.pi * dow / 7)
    df["dow_cos"] = np.cos(2 * np.pi * dow / 7)
    df["month_sin"] = np.sin(2 * np.pi * df["FL_MONTH"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["FL_MONTH"] / 12)
    del date, dow  
    R = 6371.0
    lat1, lon1 = np.radians(df["O_LATITUDE"]), np.radians(df["O_LONGITUDE"])
    lat2, lon2 = np.radians(df["D_LATITUDE"]), np.radians(df["D_LONGITUDE"])
    a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
    df["great_circle_km"] = R * 2 * np.arcsin(np.sqrt(a))
    del lat1, lon1, lat2, lon2, a

    df["dep_hour_bucket"] = (df["CRS_DEP_TIME_MIN"] // 120).astype("int8")
    congestion = (
        df.groupby(["ORIGIN_INDEX", "FL_YEAR", "FL_MONTH", "FL_DAY", "dep_hour_bucket"], observed=True)
        .size().rename("origin_congestion_2h").reset_index()
    )
    df = df.merge(congestion, on=["ORIGIN_INDEX", "FL_YEAR", "FL_MONTH", "FL_DAY", "dep_hour_bucket"], how="left")
    del congestion

    df["ARR_DELAY_BIN"] = (df["ARR_DELAY"] > delay_threshold).astype("int8")
    df["DEP_DELAY_BIN"] = (df["DEP_DELAY"] > delay_threshold).astype("int8")

    return df


### New functions

In [ ]:
# new tensor in order: FL_YEAR, FL_DAY, 'CRS_DEP_TIME_MIN', 'CRS_ARR_TIME_MIN', o lat, o long, d lat, d long, FL_WEEK, the 10 engineered features

In [ ]:
# features used in MLP: (32):
# 
# 'FL_DAY',  # day of the month - missing

# 'FL_WEEK', # week of the year - missing


# 'dep_hour_sin', 'dep_hour_cos', 'arr_hour_sin', 'arr_hour_cos', 'dow_sin', # calc from others
# 'dow_cos', 'month_sin', 'month_cos', 'great_circle_km', 'origin_congestion_2h',
# 'CRS_DEP_TIME_MIN', 'CRS_ARR_TIME_MIN', # can be converted
# # 'CRS_ELAPSED_TIME', 'FLIGHTS', can be deduced
# #  'O_LATITUDE', 'O_LONGITUDE', 'D_LATITUDE', 'D_LONGITUDE',  # can be deduced
#  'FL_YEAR', # can be deduced
# 'FL_MONTH', 
# 'ORIGIN_INDEX', 'DEST_INDEX',
# 'O_TEMP', 'O_PRCP', 'O_WSPD', 'D_TEMP', 'D_PRCP', 'D_WSPD',
# 'OP_CARRIER', 'OP_CARRIER_FL_NUM',

In [ ]:
# features in chains:
# 'DAY_OF_WEEK', 'CRS_ARR_TIME_HOUR', 'CRS_DEP_TIME_HOUR',

# 'MONTH', 
# 'ORIGIN_INDEX', 'DEST_INDEX',
# 'OP_CARRIER', 'OP_CARRIER_FL_NUM'
# 'O_TEMP', 'D_TEMP', 'O_PRCP', 'D_PRCP', 'O_WSPD', 'D_WSPD'

In [ ]:


MAX_SEQ_LEN = 6

DENSE_FEAT_COLS = ["O_TEMP", "D_TEMP", "O_PRCP", "D_PRCP", "O_WSPD", "D_WSPD"]
SPARSE_FEAT_COLS = ["MONTH_0", "DOW_0", "CRS_ARR_TIME_HOUR", "CRS_DEP_TIME_HOUR",
                     "ORIGIN_INDEX", "DEST_INDEX", "OP_CARRIER", "OP_CARRIER_FL_NUM"]
TARGET_COLS = ["ARR_DELAY", "DEP_DELAY"]        # numeric minutes -> "delays" tensor
LABEL_COLS = ["ARR_DELAY_BIN", "DEP_DELAY_BIN"]  # -> "labels" tensor (bug-fixed vs. original)

MLP_ONLY_COLS = [
    "FL_YEAR", "FL_DAY", "FL_WEEK",
    "CRS_DEP_TIME_MIN", "CRS_ARR_TIME_MIN",
    "O_LATITUDE", "O_LONGITUDE", "D_LATITUDE", "D_LONGITUDE",
    "dep_hour_sin", "dep_hour_cos", "arr_hour_sin", "arr_hour_cos",
    "dow_sin", "dow_cos", "month_sin", "month_cos",
    "great_circle_km", "origin_congestion_2h",
]  # order matters -- treat as metadata for the 6th tensor's columns


# ---------------------------------------------------------------------------
# STEP 1 -- common preprocessing (shared by both modes)
# ---------------------------------------------------------------------------
def prepare_dataframe_common(df):
    """
    ASSUMES df is the raw tabular CSV (freshly read via pd.read_csv, NOT yet
    cleaned). Runs clean_dataframe(), derives all columns needed by both the
    LSTM chain builder and the MLP feature set, then calls add_all_features().

    #  The column-derivation block below is INFERRED -- I don't have the exact
    # script that originally produced FL_YEAR/FL_MONTH/FL_DAY/CRS_DEP_TIME_MIN in
    # your pipeline. Verify against your actual notebook.
    """
    df = clean_dataframe(df)  # existing function: dtype conversions, drop NaNs

    # --- date parts (FL_DATE itself is kept, not dropped, so chain grouping
    #     can still use it directly as before) ---
    df["FL_YEAR"] = df["FL_DATE"].dt.year.astype("int16")
    df["FL_MONTH"] = df["FL_DATE"].dt.month.astype("int8")
    df["FL_DAY"] = df["FL_DATE"].dt.day.astype("int8")
    df["FL_WEEK"] = df["FL_DATE"].dt.isocalendar().week.astype("int8")

    # --- scheduled times: minutes since midnight + hour bucket ---
    df["CRS_DEP_TIME_MIN"] = (df["CRS_DEP_TIME"].dt.hour * 60 + df["CRS_DEP_TIME"].dt.minute).astype("int16")
    df["CRS_ARR_TIME_MIN"] = (df["CRS_ARR_TIME"].dt.hour * 60 + df["CRS_ARR_TIME"].dt.minute).astype("int16")
    df["CRS_DEP_TIME_HOUR"] = df["CRS_DEP_TIME"].dt.hour.astype("int8")
    df["CRS_ARR_TIME_HOUR"] = df["CRS_ARR_TIME"].dt.hour.astype("int8")

    # --- delays as numeric minutes (needed for binary targets + 'delays' tensor) ---
    df["ARR_DELAY"] = (df["ARR_DELAY"].dt.total_seconds() / 60).astype("int16")
    df["DEP_DELAY"] = (df["DEP_DELAY"].dt.total_seconds() / 60).astype("int16")

    # --- 0-indexed categorical codes, ONLY for LSTM embeddings (kept separate
    #     from raw MONTH/DAY_OF_WEEK so the MLP export can use the 1-indexed
    #     originals via FL_MONTH / cyclic features instead) ---
    df["MONTH_0"] = (df["MONTH"] - 1).astype("int16")
    df["DOW_0"] = (df["DAY_OF_WEEK"] - 1).astype("int16")

    # --- carrier / flight number as integer codes ---
    # NOTE: fit on the WHOLE dataframe before splitting -- pre-existing minor
    # leak already present in prepare_data(), not new here.
    df["OP_CARRIER"] = LabelEncoder().fit_transform(df["OP_CARRIER"]).astype("int16")
    df["OP_CARRIER_FL_NUM"] = LabelEncoder().fit_transform(df["OP_CARRIER_FL_NUM"]).astype("int16")

    # --- engineered MLP features + binary targets (your existing function) ---
    df = add_all_features(df, delay_threshold=15)

    return df


def compute_day_splits(df, seed=42, train_frac=0.6, valid_frac=0.2):
    """Assigns each unique flight DAY to train/val/test (mirrors prepare_data())."""
    date_dim = df[["FL_YEAR", "FL_MONTH", "FL_DAY"]].drop_duplicates().reset_index(drop=True)
    date_dim["DAY_OF_YEAR"] = pd.to_datetime(
        dict(year=date_dim["FL_YEAR"], month=date_dim["FL_MONTH"], day=date_dim["FL_DAY"])
    ).dt.dayofyear

    rng = np.random.RandomState(seed=seed)
    train_days, valid_days, test_days = [], [], []

    for month in sorted(date_dim["FL_MONTH"].unique()):
        month_dates = date_dim[date_dim["FL_MONTH"] == month]["DAY_OF_YEAR"]
        n_days = len(month_dates)
        if n_days < 3:
            raise Exception(f"Fewer than 3 days for month {month} -- can't guarantee a 3-way split.")
        mandatory_days = rng.choice(month_dates, 3, replace=False)
        train_days.append(mandatory_days[0])
        valid_days.append(mandatory_days[1])
        test_days.append(mandatory_days[2])

        remaining_days = [d for d in month_dates if d not in mandatory_days]
        n_remaining = len(remaining_days)
        if n_remaining > 0:
            permuted = rng.permutation(remaining_days)
            split1 = int(round(n_remaining * train_frac))
            split2 = split1 + int(round(n_remaining * valid_frac))
            train_days.extend(permuted[:split1])
            valid_days.extend(permuted[split1:split2])
            test_days.extend(permuted[split2:])

    date_dim["SPLIT_TYPE"] = np.select(
        [date_dim["DAY_OF_YEAR"].isin(train_days),
         date_dim["DAY_OF_YEAR"].isin(valid_days),
         date_dim["DAY_OF_YEAR"].isin(test_days)],
        ["train", "val", "test"],
        default="undefined",
    )
    return date_dim[["FL_YEAR", "FL_MONTH", "FL_DAY", "SPLIT_TYPE"]]


def split_by_day(df, seed=42, train_frac=0.6, valid_frac=0.2):
    day_splits = compute_day_splits(df, seed=seed, train_frac=train_frac, valid_frac=valid_frac)
    return df.merge(day_splits, on=["FL_YEAR", "FL_MONTH", "FL_DAY"], how="left")


# ---------------------------------------------------------------------------
# STEP 2a -- "full" mode: build chains -> 6-tensor PyTorch datasets
# ---------------------------------------------------------------------------
def create_flight_chains(df):
    df_sorted = df.sort_values(by=["OP_CARRIER", "OP_CARRIER_FL_NUM", "FL_DATE", "CRS_DEP_TIME"])
    grouped = df_sorted.groupby(["OP_CARRIER", "OP_CARRIER_FL_NUM", "FL_DATE"])
    return {name: group for name, group in grouped}


def adjust_sequence(data, max_len=MAX_SEQ_LEN):
    if len(data) < max_len:
        pad_shape = (max_len - len(data), data.shape[1])
        return torch.cat([data, torch.zeros(pad_shape, dtype=data.dtype)], dim=0)
    return data[:max_len]


def process_all_chains_full(flight_chains, max_sequence_length=MAX_SEQ_LEN):
    """First 5 tensors are EXACTLY the existing LSTM format; 6th is new (MLP feats)."""
    processed = []
    for name, chain in flight_chains.items():
        dense_feat = torch.stack(
            [torch.tensor(chain[c].fillna(0).values, dtype=torch.float32) for c in DENSE_FEAT_COLS],
            dim=1,
        )
        sparse_feat = torch.stack(
            [torch.tensor(chain[c].values.astype(np.int16), dtype=torch.int16) for c in SPARSE_FEAT_COLS],
            dim=1,
        )
        labels = torch.tensor(chain[LABEL_COLS].values.astype(np.int8), dtype=torch.int8)
        delays = torch.tensor(chain[TARGET_COLS].values.astype(np.int16), dtype=torch.int16)
        mlp_feat = torch.tensor(chain[MLP_ONLY_COLS].fillna(0).values, dtype=torch.float32)

        valid_len = min(len(chain), max_sequence_length)

        dense_feat = adjust_sequence(dense_feat, max_sequence_length)
        sparse_feat = adjust_sequence(sparse_feat, max_sequence_length)
        labels = adjust_sequence(labels, max_sequence_length)
        delays = adjust_sequence(delays, max_sequence_length)
        mlp_feat = adjust_sequence(mlp_feat, max_sequence_length)

        processed.append((dense_feat, sparse_feat, labels, valid_len, delays, mlp_feat))
    return processed


def create_dataset_full(processed_data):
    dense = torch.stack([item[0] for item in processed_data])
    sparse = torch.stack([item[1] for item in processed_data])
    labels = torch.stack([item[2] for item in processed_data])
    valid_lens = torch.tensor([item[3] for item in processed_data], dtype=torch.long)
    delays = torch.stack([item[4] for item in processed_data])
    mlp_feat = torch.stack([item[5] for item in processed_data])
    return TensorDataset(dense, sparse, labels, valid_lens, delays, mlp_feat)


# ---------------------------------------------------------------------------
# STEP 2b -- "mlp" mode: flight-level parquet export
# ---------------------------------------------------------------------------
MLP_EXPORT_EXTRA_EXCLUDE = {
    # raw datetime/timedelta/string columns superseded by derived features
    "CRS_DEP_TIME", "CRS_ARR_TIME", "DEP_TIME", "ARR_TIME", "WHEELS_OFF", "WHEELS_ON",
    "TAXI_OUT", "TAXI_IN", "CRS_ELAPSED_TIME", "ACTUAL_ELAPSED_TIME", "AIR_TIME",
    "ORIGIN", "DEST",
    # raw 1-indexed / LSTM-only helper columns, superseded by FL_MONTH/FL_DAY/FL_WEEK
    "MONTH", "DAY_OF_MONTH", "DAY_OF_WEEK", "MONTH_0", "DOW_0",
    # bookkeeping, not a feature
    "SPLIT_TYPE",
}


def export_mlp_parquets(df, output_dir="mlp_data"):
    exclude_cols = {"ARR_DELAY", "DEP_DELAY", "ARR_DELAY_BIN", "DEP_DELAY_BIN",
                     "FL_DATE", "dep_hour_bucket"} | MLP_EXPORT_EXTRA_EXCLUDE
    target_cols = ["ARR_DELAY_BIN", "DEP_DELAY_BIN"]
    feature_cols = [c for c in df.columns if c not in exclude_cols]

    os.makedirs(output_dir, exist_ok=True)
    paths = {}
    for split in ["train", "val", "test"]:
        split_df = df.loc[df["SPLIT_TYPE"] == split, feature_cols + target_cols]
        out_path = os.path.join(output_dir, f"mlp_{split}.parquet")
        split_df.to_parquet(out_path, index=False)
        paths[split] = out_path
        print(f"Saved {split}: {len(split_df)} rows -> {out_path}")
    return paths


# ---------------------------------------------------------------------------
# TOP-LEVEL ENTRY POINT
# ---------------------------------------------------------------------------
def prepare_multi_head_dataset(
    df, mode="full", seed=42, train_frac=0.6, valid_frac=0.2,
    max_sequence_length=MAX_SEQ_LEN, output_dir="prepared_data",
):
    df = prepare_dataframe_common(df)
    df = split_by_day(df, seed=seed, train_frac=train_frac, valid_frac=valid_frac)

    if mode == "mlp":
        return export_mlp_parquets(df, output_dir=output_dir)

    elif mode == "full":
        datasets = {}
        for split in ["train", "val", "test"]:
            split_df = df[df["SPLIT_TYPE"] == split]
            chains = create_flight_chains(split_df)
            processed = process_all_chains_full(chains, max_sequence_length=max_sequence_length)
            datasets[split] = create_dataset_full(processed)
            print(f"{split}: {len(datasets[split])} chains")
        return datasets

    else:
        raise ValueError(f"Unknown mode: {mode!r}")

### New functions tests / EXAMPLE USE

In [ ]:
# define column names of original tabular dataset as global variables
# (for dataset download later)
DATE_COLS=["FL_DATE"]
DATETIME_COLS=["CRS_DEP_TIME", "CRS_ARR_TIME", "DEP_TIME",
               "ARR_TIME", "WHEELS_OFF", "WHEELS_ON", ]
TIMEDELTA_MINS_COLS=["DEP_DELAY", "ARR_DELAY", "TAXI_OUT", "TAXI_IN", "CRS_ELAPSED_TIME",
                     "ACTUAL_ELAPSED_TIME", "AIR_TIME",	]
INT_COLS=["OP_CARRIER_FL_NUM", "FLIGHTS", "MONTH", "DAY_OF_MONTH",
          "DAY_OF_WEEK", "ORIGIN_INDEX", "DEST_INDEX"]
STR_COLS=["OP_CARRIER", "ORIGIN", "DEST"]
FLOAT_COLS=["O_TEMP", "O_PRCP", "O_WSPD", "D_TEMP", "D_PRCP", "D_WSPD", "O_LATITUDE",
             "O_LONGITUDE", "D_LATITUDE", "D_LONGITUDE"]
size_set_check=set(DATE_COLS+DATETIME_COLS+TIMEDELTA_MINS_COLS+INT_COLS+STR_COLS+FLOAT_COLS)
print(f"Total individual features (should be 34): {len(size_set_check)}")

Total individual features (should be 34): 34


In [ ]:
path_to_2022_csv, path_to_2023_csv =download_dataset(2022, 2024)

100%|██████████| 1.66G/1.66G [00:14<00:00, 121MB/s] 


100%|██████████| 1.72G/1.72G [00:15<00:00, 117MB/s] 


In [ ]:
df_2022 = pd.read_csv(path_to_2022_csv, nrows=50000) # IMPORTANT NOTE: nrows param only for testing
df_2023 = pd.read_csv(path_to_2023_csv, nrows=50000)


In [28]:
df_small = pd.concat([df_2022, df_2023], ignore_index=True)

# df_small["FL_DATE"] = pd.to_datetime(df_small["FL_DATE"])
# first_days = sorted(df_small["FL_DATE"].unique())[:40] 
# df_small = df_small[df_small["FL_DATE"].isin(first_days)].copy()


In [ ]:

datasets = prepare_multi_head_dataset(df_small, mode="full") # output: full pytorch dataset for MLP+LSTM


Dropped 5 rows because of NaNs.
Any NaNs remaining in numerical data: False.
train: 78533 chains
val: 14807 chains
test: 6655 chains


In [ ]:
sample = datasets["train"][1]
print(len(sample))          # should be 6 now, not 5
print(sample[5].shape)      # should be [6, 19] -- the new MLP tensor


6
torch.Size([6, 19])
Dropped 0 rows because of NaNs.
Any NaNs remaining in numerical data: False.


Saved train: 78533 rows -> mlp_test_out/mlp_train.parquet
Saved val: 14807 rows -> mlp_test_out/mlp_val.parquet
Saved test: 6655 rows -> mlp_test_out/mlp_test.parquet


In [ ]:

paths = prepare_multi_head_dataset(df_small, mode="mlp", output_dir="mlp_test_out") # output: path to parquet train / val / test files for MLP

In [ ]:
for i in range(6): 
    print(f"--- tensors[{i}] ---")
    print(sample[i])

# IMPORTANT NOTE: in the last tensor, each of the 6 arrays (one per flight) collects the 19 features for MLP

--- tensors[0] ---
tensor([[ 6.1000,  7.8000,  0.0000,  0.0000,  0.0000, 14.8000],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000]])
--- tensors[1] ---
tensor([[   2,    3,   17,   13,   22,  257,    0, 1627],
        [   0,    0,    0,    0,    0,    0,    0,    0],
        [   0,    0,    0,    0,    0,    0,    0,    0],
        [   0,    0,    0,    0,    0,    0,    0,    0],
        [   0,    0,    0,    0,    0,    0,    0,    0],
        [   0,    0,    0,    0,    0,    0,    0,    0]], dtype=torch.int16)
--- tensors[2] ---
tensor([[0, 0],
        [0, 0],
        [0, 0],
        [0, 0],
        [0, 0],
        [0, 0]], dtype=torch.int8)
--- tensors[3] ---
tensor(1)
--- tensors[4] ---
tensor([[-22,  -4],
       

In [ ]:
# desired output:
# Sample Type: <class 'tuple'>
# Total components in the sample tuple: 5

# ------------------------------------------------------------
# 1. Dense Features (Continuous / Meteorological Features):
#    - Shape: torch.Size([6, 6]) (Sequence Length x 6)
#    - Cols: ['O_TEMP', 'D_TEMP', 'O_PRCP', 'D_PRCP', 'O_WSPD', 'D_WSPD']
#     - Values:
# tensor([[ 6.7000, 19.4000,  0.0000,  0.0000,  9.4000, 14.8000],
#         [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
#         [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
#         [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
#         [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
#         [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000]])

# 2. Sparse Features (Categorical / Temporal Features):
#    - Shape: torch.Size([6, 8]) (Sequence Length x 8)
#    - Cols: ['MONTH', 'DAY_OF_WEEK', 'CRS_ARR_TIME_HOUR', 'CRS_DEP_TIME_HOUR', 'ORIGIN_INDEX', 'DEST_INDEX', 'OP_CARRIER', 'OP_CARRIER_FL_NUM']
#     - Values:
# tensor([[   0,    5,   11,    9,   93,  257,    0, 4623],
#         [   0,    0,    0,    0,    0,    0,    0,    0],
#         [   0,    0,    0,    0,    0,    0,    0,    0],
#         [   0,    0,    0,    0,    0,    0,    0,    0],
#         [   0,    0,    0,    0,    0,    0,    0,    0],
#         [   0,    0,    0,    0,    0,    0,    0,    0]], dtype=torch.int16)

# 3. Binary Labels (Flight Delay Indicators > 15 mins):
#    - Shape: torch.Size([6, 2]) (Sequence Length x 2)
#    - [(ARR_DELAY > 15), (DEP_DELAY > 15)]
#    - Values:
# tensor([[0, 0],
#         [0, 0],
#         [0, 0],
#         [0, 0],
#         [0, 0],
#         [0, 0]], dtype=torch.int8)

# 4. Valid Sequence Lengths (Metadata):
#    - Description: Effective number of valid flights in the chain before padding
#    - Shape: torch.Size([])
#    - Values: 1

# 5. Raw Delays (Ground Truth):
#    - Shape: torch.Size([6, 2]) (Sequence Length x 2)
#    - Cols: ['ARR_DELAY', 'DEP_DELAY']
#    - Values:
# tensor([[-29,  -5],
#         [  0,   0],
#         [  0,   0],
#         [  0,   0],
#         [  0,   0],
#         [  0,   0]], dtype=torch.int16)
# ------------------------------------------------------------

# 6. PLUS the last tensor

In [ ]:
def check_parquet_files(output_dir="mlp_test_out"):
    for split in ["train", "val", "test"]:
        file_path = os.path.join(output_dir, f"mlp_{split}.parquet")
        if not os.path.exists(file_path):
            print(f" {split} file missing: {file_path}")
            continue
        df = pd.read_parquet(file_path)
        print(f"\n{'='*60}")
        print(f"SPLIT: {split.upper()}")
        print(f"Shape: {df.shape}")
        print(f"Columns ({len(df.columns)}): {list(df.columns)}")
        print(f"\n--- Missing values ---")
        print(df.isna().sum()[df.isna().sum() > 0])
        if df.isna().sum().sum() > 0:
            print("  There are missing values!")
        else:
            print(" No missing values.")
        
        # Target columns
        target_cols = ["ARR_DELAY_BIN", "DEP_DELAY_BIN"]
        if all(c in df.columns for c in target_cols):
            print("\n--- Target class distribution ---")
            for col in target_cols:
                print(f"{col}: {df[col].value_counts(normalize=True).sort_index().to_dict()}")
        else:
            print("  Target columns missing!")
        
        # Check constant columns
        const_cols = [c for c in df.columns if df[c].nunique() == 1]
        if const_cols:
            print(f"  Constant columns: {const_cols}")
        
        # Summary stats for numeric columns
        num_cols = df.select_dtypes(include=[np.number]).columns
        print("\n--- Summary stats (numeric) ---")
        print(df[num_cols].describe(percentiles=[.25, .5, .75]).T[["min", "max", "mean", "std", "25%", "50%", "75%"]].head(10))
        
        # Check for extreme outliers (beyond 5 std)
        for col in num_cols:
            std = df[col].std()
            if std > 0:
                extreme = ((df[col] - df[col].mean()).abs() > 5*std).sum()
                if extreme > 0:
                    print(f"  {col}: {extreme} extreme outliers (>5 sigmas)")

check_parquet_files()


SPLIT: TRAIN
Shape: (78533, 35)
Columns (35): ['OP_CARRIER', 'OP_CARRIER_FL_NUM', 'FLIGHTS', 'ORIGIN_INDEX', 'DEST_INDEX', 'O_TEMP', 'O_PRCP', 'O_WSPD', 'D_TEMP', 'D_PRCP', 'D_WSPD', 'O_LATITUDE', 'O_LONGITUDE', 'D_LATITUDE', 'D_LONGITUDE', 'FL_YEAR', 'FL_MONTH', 'FL_DAY', 'FL_WEEK', 'CRS_DEP_TIME_MIN', 'CRS_ARR_TIME_MIN', 'CRS_DEP_TIME_HOUR', 'CRS_ARR_TIME_HOUR', 'dep_hour_sin', 'dep_hour_cos', 'arr_hour_sin', 'arr_hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'great_circle_km', 'origin_congestion_2h', 'ARR_DELAY_BIN', 'DEP_DELAY_BIN']

--- Missing values ---
Series([], dtype: int64)
 No missing values.

--- Target class distribution ---
ARR_DELAY_BIN: {0: 0.8205467765143315, 1: 0.17945322348566844}
DEP_DELAY_BIN: {0: 0.8149695032661429, 1: 0.1850304967338571}
  Constant columns: ['FLIGHTS']

--- Summary stats (numeric) ---
                          min          max        mean         std    25%  \
OP_CARRIER           0.000000    13.000000    5.424764    3.873484    2.

In [ ]:
# Then save model weights to be reused EXACTLY the same in LSTM + MLP embedding single flight's features